# RAG with HotpotQA

## Preprocessing

In [ ]:
import pandas as pd, ast

In [ ]:
def parse_doc(filename):
    df = pd.read_csv(filename)
    
    # context là chuỗi JSON của dạng {'title': [...], 'sentences': [[...],[...]]}
    def join_context(ctx_str):
        ctx = ast.literal_eval(ctx_str)
        docs = []
        for t, sents in zip(ctx["title"], ctx["sentences"]):
            if len(sents) > 0:
                docs.append({"title": t, "text": " ".join(sents)})
        return docs
    
    df["docs"] = df["context"].apply(join_context)
    return df

In [ ]:
df_train = parse_doc("/kaggle/input/hotpotqa-distractor/train.csv")
df_valid = parse_doc("/kaggle/input/hotpotqa-distractor/valid.csv")

In [ ]:
df_train.info()

In [ ]:
df_valid.info()

In [ ]:
df_train["docs"][0]

### Xây corpus

In [ ]:
# Tạo corpus (flatten toàn bộ paragraph)
corpus = []
for docs in df_train["docs"]:
    corpus.extend(docs)
corpus_texts = [c["text"] for c in corpus]


## TRAINING

In [ ]:
import transformers.utils.hub as hub

def no_check_repo_templates(*args, **kwargs):
    return []   # luôn trả về rỗng => không gọi API phụ

hub.list_repo_templates = no_check_repo_templates


In [ ]:
!pip install -q sentence-transformers torch tqdm

In [ ]:
import torch, random, ast, numpy as np
from tqdm import tqdm
from torch import nn, optim
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, util
from huggingface_hub import snapshot_download
import os
from torch.cuda.amp import autocast, GradScaler

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

### Data Loader

In [ ]:
train_examples = []
for _, row in df_train.iterrows():
    ctx = row["docs"]
    pos_titles = ast.literal_eval(row["supporting_facts"])["title"]
    pos_docs = [d["text"] for d in ctx if d["title"] in pos_titles]
    neg_docs = [d["text"] for d in ctx if d["title"] not in pos_titles]
    if not pos_docs or not neg_docs:
        continue
    pos = random.choice(pos_docs)
    train_examples.append(InputExample(texts=[row["question"], pos]))

print("Total training pairs:", len(train_examples))


In [ ]:
batch_size = 16
train_loader = DataLoader(train_examples, 
                          batch_size=batch_size, 
                          shuffle=True, 
                          drop_last=True,
                          collate_fn=lambda x: x )

### Model

In [ ]:
def load_or_download_model(repo_id, local_path, force_download=False):

    if not os.path.exists(local_path) or force_download:
        model_path = snapshot_download(repo_id)
        os.environ["HF_HUB_OFFLINE"] = "1"
        model = SentenceTransformer(model_path, local_files_only=True)
        model.save(e5_path)
        print(f"✅ Saved snapshot to {local_path}")
        return model
    else:
        os.environ["HF_HUB_OFFLINE"] = "1"
        model = SentenceTransformer(local_path, local_files_only=True)

    return model
        


In [ ]:
def train_retriever(
    train_loader,
    model,
    lr=2e-5,
    epochs=20,
    patience=3,
    temperature=20.0,
    output_path="./retriever",
    accumulation_steps=2,   # ✅ chia nhỏ batch để tiết kiệm VRAM
    use_fp16=True,          # ✅ bật mixed precision
):
    """
    Generic training loop for bi-encoder retriever models using
    MultipleNegativesRankingLoss (InfoNCE) with early stopping.

    Args:
        train_loader: DataLoader yielding InputExample batches
        model: SentenceTransformer model (bi-encoder)
        lr: learning rate
        epochs: max epochs
        patience: early stopping patience
        temperature: similarity scaling
        output_path: where to save best checkpoint
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    scaler = GradScaler(enabled=use_fp16)

    best_loss = float("inf")
    patience_counter = 0

    # optional: clamp max_seq_length để tránh OOM
    if hasattr(model, "max_seq_length"):
        model.max_seq_length = min(model.max_seq_length, 128)

    for epoch in range(epochs):
        model.train()
        epoch_losses = []
        progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        
        optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(progress):
            # unpack
            questions = [ex.texts[0] for ex in batch]
            passages  = [ex.texts[1] for ex in batch]

            # encode
            # 1️⃣ Tokenize (vẫn nằm trong graph)
            q_features = model.tokenize(questions)
            p_features = model.tokenize(passages)
        
            # 2️⃣ Move tensors to GPU
            q_features = {k: v.to(device) for k, v in q_features.items()}
            p_features = {k: v.to(device) for k, v in p_features.items()}
        
            # 3️⃣ Forward → lấy embedding có requires_grad=True
            with autocast(enabled=use_fp16):
                q_emb = model(q_features)["sentence_embedding"]
                p_emb = model(p_features)["sentence_embedding"]

                sim = torch.matmul(q_emb, p_emb.T) * temperature
                labels = torch.arange(sim.size(0)).to(device)
                loss = loss_fn(sim, labels) / accumulation_steps
                
            # similarity + loss
            scaler.scale(loss).backward()
            if (step + 1) % accumulation_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                torch.cuda.empty_cache()  # 🔧 clear fragmented VRAM

            loss_val = loss.item()
            epoch_losses.append(loss_val)
            progress.set_postfix({"batch_loss": f"{loss_val:.4f}"})
    
        avg_loss = np.mean(epoch_losses)
        print(f"📉 Epoch {epoch+1}: avg_loss={avg_loss:.4f}")

        # early stopping
        if avg_loss < best_loss - 1e-4:
            best_loss = avg_loss
            patience_counter = 0
            model.save(output_path)
            print(f"✅ Saved new best model → {output_path}")
        else:
            patience_counter += 1
            print(f"⚠️ No improvement ({patience_counter}/{patience})")
            if patience_counter >= patience:
                print("🛑 Early stopping triggered")
                break

    print(f"🎯 Training done. Best loss={best_loss:.4f}")
    return model


**Load Model**

In [ ]:
e5_path = "/kaggle/working/retriever/e5-base-v2"
mini_path = "/kaggle/working/retriever/all-MiniLM-L6-v2"
bge_path = "/kaggle/working/retriever/bge-small-en-v1.5"

In [ ]:
e5 = load_or_download_model("intfloat/e5-base-v2", e5_path)
miniLM = load_or_download_model("sentence-transformers/all-MiniLM-L6-v2", mini_path)
bge = load_or_download_model("BAAI/bge-small-en-v1.5", bge_path)

In [ ]:
# e5-base-v2
train_retriever(train_loader, e5, lr=2e-5, epochs=5,
                     output_path=e5_path)

In [ ]:
# all-MiniLM-L6-v2
train_retriever(train_loader, e5, lr=2e-5, epochs=5,
                     output_path=mini_path)

In [ ]:
#bge-small-en-v1.5
train_retriever(train_loader, e5, lr=2e-5, epochs=5,
                     output_path=bge_path)

### Evaluate

In [ ]:
def evaluate_retriever(
    df_valid,
    model_path,
    top_k=5,
    cache_path="/kaggle/working/corpus_embeds.pt",
    rebuild_cache=False,
    device=None,
    batch_size=64,
    verbose=True,
):
    """
    Evaluate retriever on large dataset by pre-encoding corpus embeddings.

    Args:
        df_valid: DataFrame with ["question", "docs", "supporting_facts"]
        model_path: path to retriever (fine-tuned or base)
        top_k: number of retrieved docs to check
        cache_path: where to store/reuse precomputed doc embeddings
        rebuild_cache: force re-encode corpus even if cache exists
        device: 'cuda' or 'cpu'
        batch_size: encoding batch size
        verbose: show progress

    Returns:
        metrics dict (top1_acc, top3_acc, top5_acc, mrr)
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # 🧩 Load model
    model = SentenceTransformer(model_path, device=device)
    model.eval()

    # 🧱 Build corpus (flatten all docs)
    all_titles, all_texts = [], []
    for row in df_valid["docs"]:
        for d in row:
            all_titles.append(d["title"])
            all_texts.append(d["text"])
    unique_corpus = dict(zip(all_titles, all_texts))  # deduplicate by title
    corpus_titles = list(unique_corpus.keys())
    corpus_texts = list(unique_corpus.values())

    # ⚡ Encode corpus (cacheable)
    if not rebuild_cache and os.path.exists(cache_path):
        if verbose: print(f"✅ Loading cached corpus embeddings from {cache_path}")
        corpus_embs = torch.load(cache_path, map_location=device)
    else:
        if verbose: print(f"📦 Encoding corpus of {len(corpus_texts)} passages...")
        corpus_embs = model.encode(
            corpus_texts, batch_size=batch_size,
            convert_to_tensor=True, normalize_embeddings=True
        )
        torch.save(corpus_embs, cache_path)
        if verbose: print(f"✅ Saved corpus embeddings → {cache_path}")

    # 🧮 Evaluation loop
    top1_hits, top3_hits, top5_hits, mrrs = [], [], [], []

    iterator = tqdm(df_valid.iterrows(), total=len(df_valid), disable=not verbose)
    for _, row in iterator:
        question = row["question"]
        gold_titles = set(
            eval(row["supporting_facts"])["title"]
            if isinstance(row["supporting_facts"], str)
            else row["supporting_facts"]["title"]
        )

        # Encode question
        q_emb = model.encode(question, convert_to_tensor=True, normalize_embeddings=True)
        sims = util.cos_sim(q_emb, corpus_embs)[0]
        top_idx = torch.topk(sims, k=top_k).indices.tolist()
        ranked_titles = [corpus_titles[i] for i in top_idx]

        # top-k hits
        top1_hits.append(any(t in gold_titles for t in ranked_titles[:1]))
        top3_hits.append(any(t in gold_titles for t in ranked_titles[:3]))
        top5_hits.append(any(t in gold_titles for t in ranked_titles[:5]))

        # MRR
        first_hit = next((i for i, t in enumerate(ranked_titles) if t in gold_titles), None)
        if first_hit is not None:
            mrrs.append(1.0 / (first_hit + 1))

    metrics = {
        "model": os.path.basename(model_path.rstrip("/")),
        "top1_acc": np.mean(top1_hits),
        "top3_acc": np.mean(top3_hits),
        "top5_acc": np.mean(top5_hits),
        "mrr": np.mean(mrrs)
    }

    if verbose:
        print(f"\n📊 {metrics['model']} Metrics:")
        for k, v in metrics.items():
            if k != "model":
                print(f"{k}: {v:.4f}")

    return metrics


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_model_comparison(results):
    """
    Plot comparison bar chart for multiple retriever models.
    Args:
        results: list of dicts, each from evaluate_retriever_cached()
    """
    df = pd.DataFrame(results)
    df_melt = df.melt(id_vars="model", var_name="metric", value_name="score")
    df_melt = df_melt[df_melt["metric"].isin(["top1_acc","top3_acc","top5_acc","mrr"])]

    plt.figure(figsize=(10,6))
    sns.barplot(data=df_melt, x="metric", y="score", hue="model")
    plt.title("🔍 Retriever Performance Comparison")
    plt.ylabel("Score")
    plt.ylim(0, 1)
    plt.legend(title="Model", loc="lower right")
    plt.grid(axis='y', linestyle='--', alpha=0.4)
    plt.show()


In [ ]:
# Evaluate 3 retrievers
metrics_e5 = evaluate_retriever(df_valid, e5_path, cache_path="/kaggle/working/corpus_e5.pt")
metrics_mini  = evaluate_retriever(df_valid, mini_path, cache_path="/kaggle/working/corpus_mpnet.pt")
metrics_bge    = evaluate_retriever(df_valid, bge_path, cache_path="/kaggle/working/corpus_bge.pt")

In [ ]:
plot_model_comparison([metrics_e5, metrics_mini, metrics_bge])